# Agent Development Notebook

Interactive notebook for testing agent nodes as they are implemented.

## Prerequisites
```bash
cd ~/devops-ai-agentic && git pull
```

In [ ]:
!pip install -r /opt/app-root/src/devops-ai-agentic/agent/requirements.txt

In [ ]:
# MUST run before any other import
import pysqlite3
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
sys.path.insert(0, "/opt/app-root/src/devops-ai-agentic")
print("Setup complete")

## Story 2.3 — RAG Knowledge Base

In [ ]:
from agent.knowledge import build_index, search_knowledge

print("Building index (downloads nomic-embed-text-v1 ~547 MB on first run)...")
build_index()
print("Index ready.")

In [ ]:
results = search_knowledge("container cannot pull image from registry", n_results=2)
for r in results:
    print(r[:400])
    print("---")

In [ ]:
results = search_knowledge("secret db-password not found", alert_type="MISSING_SECRET", n_results=1)
for r in results:
    print(r[:400])

## Story 2.4 — monitor_cluster node

In [ ]:
from agent.nodes.monitor_cluster import monitor_cluster, WATCHED_NAMESPACES

print(f"Watched namespaces: {WATCHED_NAMESPACES}")

result = monitor_cluster({})
alerts = result["alerts"]
print(f"Alerts found: {len(alerts)}")
for alert in alerts:
    print(f"  [{alert['namespace']}/{alert['pod']}] {alert['reason']}: {alert['message'][:120]}")

In [ ]:
# Deploy a broken pod to test detection
import subprocess
subprocess.run([
    "oc", "run", "broken-pod", "-n", "ai-agentic",
    "--image=registry.example.com/nonexistent:v999",
    "--restart=Never",
], check=False)
print("Broken pod created — wait ~30s then re-run the cell above")

In [ ]:
# Clean up
subprocess.run(["oc", "delete", "pod", "broken-pod", "-n", "ai-agentic"], check=False)
print("Cleaned up")

## Full Graph

## Story 2.5 — classify_alert node

In [ ]:
import os
os.environ["QWEN_INFERENCE_URL"] = "http://qwen-predictor.ai-agentic.svc.cluster.local:8080"

from agent.nodes.classify_alert import classify_alert

# Test 1 — ImagePullBackOff (fast-path, no LLM call)
state = {"current_alert": {"reason": "ImagePullBackOff", "message": "pull access denied for registry.example.com/myapp:v2"}}
result = classify_alert(state)
print(f"Test 1 ImagePullBackOff : {result['alert_type']}")

# Test 2 — Missing secret (fast-path, no LLM call)
state = {"current_alert": {"reason": "CreateContainerConfigError", "message": "secret \"db-password\" not found"}}
result = classify_alert(state)
print(f"Test 2 MissingSecret    : {result['alert_type']}")

# Test 3 — Ambiguous alert (LLM call to Qwen)
state = {"current_alert": {"reason": "CrashLoopBackOff", "message": "container exited with code 1"}}
result = classify_alert(state)
print(f"Test 3 CrashLoopBackOff : {result['alert_type']}")

In [ ]:
from agent.graph import graph
from IPython.display import Image

Image(graph.get_graph().draw_mermaid_png())

In [ ]:
# Run the full graph (uncomment when all nodes are implemented)
# result = graph.invoke({})
# print(result.get("report", "No report generated"))